In [11]:
import sys
import os
import copy
import random
from pathlib import Path

current_dir = Path.cwd()
script_dir = Path(__file__).resolve().parent if '__file__' in globals() else current_dir

def find_repo_root(start: Path) -> Path | None:
    for p in [start, *start.parents]:
        if (p / 'pyproject.toml').exists() and (p / 'bcosgnn').is_dir():
            return p
    return None

project_root = find_repo_root(current_dir) or find_repo_root(script_dir)

if project_root is not None:
    if str(project_root) not in sys.path:
        sys.path.append(str(project_root))
    print(f"Repo root added: {project_root}")
else:
    print("Repo root not found in this execution context; using installed packages and local paths.")
    project_root = current_dir

default_data_root = project_root / 'data' / 'MNISTsp'
data_root = Path(os.environ.get('MNISTSP_DATA_ROOT', str(default_data_root))).expanduser()
print(f"Using data root: {data_root}")

Repo root added: /Users/shaique/Desktop/BioInf_IMP/NMM_group/ICML/bcos_gnn/bcosgnn
Using data root: /Users/shaique/Desktop/BioInf_IMP/NMM_group/ICML/bcos_gnn/bcosgnn/data/MNISTsp


In [12]:
import bcosgnn
import torch
import torch_geometric
import functools
import itertools
import operator
from typing import Any
import torch
from torch_geometric.data import Dataset, download_url
from torch.utils.data import random_split
import numpy as np
import polars as pl
import seaborn as sns
import matplotlib.pyplot as plt
import torch
from bcos.modules import BcosLinear, BcosSequential
from sklearn.model_selection import train_test_split
from torch.nn import BCEWithLogitsLoss
from torch_geometric.datasets import BA2MotifDataset
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import MessagePassing
from torch_geometric.nn.aggr import SumAggregation
from torch_geometric.utils import add_self_loops, degree
from torchmetrics import AUROC
from torchmetrics.classification import BinaryAccuracy
from tqdm import tqdm
import networkx as nx
import matplotlib.pyplot as plt
import torch.nn.functional as F
from bcosgnn.explain import explain
from tqdm import tqdm
from torch_geometric.datasets import MNISTSuperpixels
import torch.nn as nn
from torch_geometric.nn import GINEConv, global_mean_pool, global_add_pool
from sklearn.metrics import f1_score, accuracy_score
from torch.utils.data import random_split

import time
from sklearn.metrics import roc_auc_score, f1_score
from bcosgnn.explain import explain


In [13]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

In [14]:

def _augment_with_normalized_pos(dataset):
    augmented = []
    for data in dataset:
        d = data.clone()
        pos = d.pos.float()
        pos_min = pos.min(dim=0).values
        pos_max = pos.max(dim=0).values
        denom = (pos_max - pos_min).clamp(min=1e-8)
        pos_norm = (pos - pos_min) / denom
        d.x = torch.cat([d.x.float(), pos_norm], dim=-1)
        augmented.append(d)
    return augmented

def load_and_split_data(batch_size=128, add_pos_features=True):
    print("Loading preprocessed sparsified .pt splits...")
    candidate_roots = [
        project_root / "data" / "MNIST" / "sparsified_pt_splits",
        project_root / "shaique_updates" / "codes" / "MNISTsp" / "data" / "MNIST" / "sparsified_pt_splits",
    ]

    split_root = None
    for root in candidate_roots:
        if (root / "train_sparsified.pt").exists() and (root / "val_sparsified.pt").exists() and (root / "test_sparsified.pt").exists():
            split_root = root
            break
    if split_root is None:
        raise FileNotFoundError("Could not find saved sparsified split files.")

    train_dataset = torch.load(split_root / "train_sparsified.pt", map_location="cpu", weights_only=False)
    val_dataset = torch.load(split_root / "val_sparsified.pt", map_location="cpu", weights_only=False)
    test_dataset = torch.load(split_root / "test_sparsified.pt", map_location="cpu", weights_only=False)

    # Balance test_dataset to 1000 datapoints (100 per class)
    class_counts = {i: 0 for i in range(10)}
    balanced_test = []
    for data in test_dataset:
        label = int(data.y.item())
        if class_counts.get(label, 0) < 100:
            balanced_test.append(data)
            class_counts[label] = class_counts.get(label, 0) + 1
    test_dataset = balanced_test

    if add_pos_features:
        train_dataset = _augment_with_normalized_pos(train_dataset)
        val_dataset = _augment_with_normalized_pos(val_dataset)
        test_dataset = _augment_with_normalized_pos(test_dataset)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    all_labels = torch.tensor([int(d.y.item()) for d in train_dataset + val_dataset + test_dataset])
    num_classes = int(all_labels.unique().numel())
    num_node_features = int(train_dataset[0].num_node_features)
    num_edge_features = int(train_dataset[0].edge_attr.size(1)) if hasattr(train_dataset[0], "edge_attr") and train_dataset[0].edge_attr is not None else 0

    dataset_info = {
        "num_node_features": num_node_features,
        "num_edge_features": num_edge_features,
        "num_classes": num_classes,
    }

    print(f"Using split directory: {split_root}")
    print(f"Loaded splits. Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")
    print(f"Node features: {num_node_features} (pos features added: {add_pos_features})")
    
    # Store full_dataset mock for compatibility with previous code if needed
    class MockFullDataset:
        def __init__(self, node_dim, edge_dim, num_classes):
            self.num_node_features = node_dim
            self.num_edge_features = edge_dim
            self.num_classes = num_classes
    full_dataset = MockFullDataset(num_node_features, num_edge_features, num_classes)

    return train_loader, val_loader, test_loader, full_dataset, test_dataset


## BCos model Definitions

In [15]:
class Readout(torch.nn.Module):
    def __init__(
        self,
        in_channels,
        hidden_channels=None,
        out_channels=1,
        b=2,
        max_out=1,
        agg: str = "sum",
    ):
        super().__init__()
        if hidden_channels is None:
            self.readout = BcosLinear(in_channels, out_channels, b=b, max_out=max_out)
        else:
            hidden_channels = (
                [hidden_channels]
                if isinstance(hidden_channels, int)
                else hidden_channels
            )
            channels = [in_channels] + hidden_channels + [out_channels]
            self.readout = BcosSequential(
                *[
                    BcosLinear(d_in, d_out, b=b, max_out=max_out)
                    for d_in, d_out in zip(channels[:-1], channels[1:])
                ]
            )
        match agg:
            case "sum":
                self.agg = SumAggregation()
            case _:
                raise ValueError(f"Aggregation '{agg}' not supported.")

    def forward(self, x, batch):
        raise NotImplementedError


class AggThenReadout(Readout):
    def forward(self, x, batch):
        z = self.agg(x, batch)
        out = self.readout(z)
        return out

class ReadoutThenAgg(Readout):
    def forward(self, x, batch):
        z = self.readout(x)
        out = self.agg(z, batch)
        return out

import torch
from torch_geometric.nn import MessagePassing

class BcosGINEConv(MessagePassing):
    def __init__(
        self,
        channels: list[int],
        edge_dim: int,
        b: float = 2.0,
        max_out: int = 1,
        eps: float = 0.0,
        train_eps: bool = False,
        **kwargs
    ):
        # We use 'add' aggregation to stay true to the GIN formula
        kwargs.setdefault("aggr", "add")
        super().__init__(**kwargs)
        
        # The MLP part of GIN, but using B-cos layers
        self.transform = BcosSequential(
            *[
                BcosLinear(din, dout, b=b, max_out=max_out)
                for din, dout in zip(channels[:-1], channels[1:])
            ]
        )
        
        self.initial_eps = eps
        if train_eps:
            self.eps = torch.nn.Parameter(torch.Tensor([eps]))
        else:
            self.register_buffer("eps", torch.Tensor([eps]))

    def forward(self, x, edge_index, edge_attr):
        # edge_attr is expected to be pre-projected to match x dimension
        # in your main model loop.
        
        # 1. Propagate messages
        out = self.propagate(edge_index, x=x, edge_attr=edge_attr)
        
        # 2. Combine step: (1 + eps) * center_node + aggregated_messages
        out = (1 + self.eps) * x + out
        
        # 3. Apply the B-cos MLP transformation
        return self.transform(out)

    def message(self, x_j, edge_attr):
        # Standard GINE uses ReLU(x_j + edge_attr).
        # In pure B-cos, we can use the addition, then the B-cos transform 
        # in the 'forward' call handles the non-linear alignment.
        return torch.nn.functional.relu(x_j + edge_attr)

In [16]:
class PureBcosGINE(nn.Module):
    def __init__(
        self,
        node_dim: int,
        edge_dim: int,
        hidden_dim: int = 64,
        num_layers: int = 4,
        num_classes: int = 9,
        b: float = 2.0,
        max_out: int = 1,
        dropout: float = 0.5,
    ):
        super().__init__()
        
        self.hidden_dim = hidden_dim
        self.has_edge_attr = edge_dim > 0
        
        self.lin_node = BcosLinear(node_dim, hidden_dim, b=b, max_out=max_out)
        self.lin_edge = BcosLinear(edge_dim, hidden_dim, b=b, max_out=max_out) if self.has_edge_attr else None
        
        self.convs = nn.ModuleList([
            BcosGINEConv(
                channels=[hidden_dim, hidden_dim],
                edge_dim=hidden_dim,
                b=b,
                max_out=max_out
            ) for _ in range(num_layers)
        ])
        
        self.readout_mlp = BcosSequential(
            BcosLinear(hidden_dim, hidden_dim, b=b, max_out=max_out),
            BcosLinear(hidden_dim, num_classes, b=b, max_out=max_out)
        )
        
        self.dropout_layer = nn.Dropout(dropout)
        self.agg = SumAggregation()

    def forward(self, x, edge_index, edge_attr, batch):
        x = self.lin_node(x)

        if self.has_edge_attr and edge_attr is not None:
            e = self.lin_edge(edge_attr)
        else:
            e = x.new_zeros((edge_index.size(1), self.hidden_dim))
        
        for conv in self.convs:
            x = conv(x, edge_index, e)
        
        node_logits = self.readout_mlp(x)
        node_logits = self.dropout_layer(node_logits)
        graph_logits = self.agg(node_logits, batch)
        
        return graph_logits

## Training & Evaluation (B-Cos GINE)
This section mirrors the vanilla MNIST notebook protocol: train/val/test loaders, early stopping, 3 random seeds, and final mean ± std reporting.

In [17]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0

    for data in tqdm(loader, desc="Training", leave=False):
        data = data.to(device)
        optimizer.zero_grad()

        out = model(data.x, data.edge_index, data.edge_attr, data.batch)
        loss = criterion(out, data.y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * data.num_graphs

    return total_loss / len(loader.dataset)


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for data in tqdm(loader, desc="Evaluating", leave=False):
            data = data.to(device)
            out = model(data.x, data.edge_index, data.edge_attr, data.batch)
            loss = criterion(out, data.y)

            total_loss += loss.item() * data.num_graphs

            pred = torch.argmax(out, dim=1)
            all_preds.append(pred.cpu().numpy())
            all_labels.append(data.y.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    y_true = np.concatenate(all_labels)
    y_pred = np.concatenate(all_preds)

    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='weighted')

    return avg_loss, acc, f1

In [ ]:

def _sync_if_cuda():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def _get_single_graph_batch(data):
    device = data.x.device
    return data.batch if hasattr(data, "batch") else torch.zeros(data.num_nodes, dtype=torch.long, device=device)

def benchmark_test_inference_total(model, test_loader, device):
    model.eval()
    _sync_if_cuda()
    t0 = time.perf_counter()
    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(device)
            _ = model(batch.x, batch.edge_index, getattr(batch, "edge_attr", None), batch.batch)
    _sync_if_cuda()
    return time.perf_counter() - t0

def evaluate_bcos_dynamic_k(model, test_loader, test_dataset, device):
    model.eval()
    correct_graphs = 0
    total_graphs = 0
    node_aurocs = []
    node_jaccards = []
    node_f1s = []
    
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(device)
            logits = model(batch.x, batch.edge_index, getattr(batch, "edge_attr", None), batch.batch)
            pred_classes = logits.argmax(dim=1)
            
            all_preds.append(pred_classes.cpu())
            all_labels.append(batch.y.cpu())
            
            correct_graphs += (pred_classes == batch.y).sum().item()
            total_graphs += batch.num_graphs
            
    all_preds_np = torch.cat(all_preds).numpy()
    all_labels_np = torch.cat(all_labels).numpy()
    # Macro F1 for multi-class MNIST
    test_f1 = f1_score(all_labels_np, all_preds_np, average='macro')

    for data in test_dataset:
        data = data.to(device)
        batch = _get_single_graph_batch(data)

        if hasattr(data, "node_mask"):
            gt_mask = data.node_mask.detach().cpu().numpy().astype(int)
        elif hasattr(data, "explanation_mask"):
            gt_mask = data.explanation_mask.detach().cpu().numpy().astype(int)
        else:
            continue

        with torch.no_grad():
            logits = model(data.x, data.edge_index, getattr(data, "edge_attr", None), batch)
            pred_label = logits.argmax(dim=-1).item()

        node_contributions = explain(
            model,
            data.x,
            data.edge_index,
            getattr(data, "edge_attr", None),
            batch,
        )

        node_attr = node_contributions.abs().sum(dim=1)

        if node_attr.max() > node_attr.min():
            node_scores = (node_attr - node_attr.min()) / (node_attr.max() - node_attr.min())
        else:
            node_scores = torch.zeros_like(node_attr)
        node_scores_np = node_scores.detach().cpu().numpy()

        if len(np.unique(gt_mask)) > 1:
            node_aurocs.append(roc_auc_score(gt_mask, node_scores_np))

        k = int(gt_mask.sum())
        if k > 0:
            _, top_k = torch.topk(node_attr, k=min(k, data.num_nodes))
            pred_binary = np.zeros(data.num_nodes, dtype=int)
            pred_binary[top_k.detach().cpu().numpy()] = 1
            intersect = (pred_binary * gt_mask).sum()
            union = (pred_binary + gt_mask).clip(0, 1).sum()
            node_jaccards.append(intersect / (union + 1e-8))
            
            # F1 Score calculation
            TP = intersect
            FP = pred_binary.sum() - TP
            FN = gt_mask.sum() - TP
            precision = TP / (TP + FP + 1e-8)
            recall = TP / (TP + FN + 1e-8)
            node_f1s.append(2 * (precision * recall) / (precision + recall + 1e-8))

    acc = correct_graphs / max(total_graphs, 1)
    node_auc = float(np.mean(node_aurocs)) if node_aurocs else float("nan")
    node_jaccard = float(np.mean(node_jaccards)) if node_jaccards else float("nan")
    node_f1 = float(np.mean(node_f1s)) if node_f1s else float("nan")
    return acc, test_f1, node_auc, node_f1, node_jaccard

def time_bcos_explanations(model, dataset, device, warmup=2):
    model.eval()
    times_ms = []
    graphs = dataset

    for data in graphs[:warmup]:
        data = data.to(device)
        batch = _get_single_graph_batch(data)
        with torch.no_grad():
            logits = model(data.x, data.edge_index, getattr(data, "edge_attr", None), batch)
            pred_label = logits.argmax(dim=-1).item()
        _ = explain(
            model,
            data.x,
            data.edge_index,
            getattr(data, "edge_attr", None),
            batch,
        )

    for data in graphs:
        data = data.to(device)
        batch = _get_single_graph_batch(data)
        with torch.no_grad():
            logits = model(data.x, data.edge_index, getattr(data, "edge_attr", None), batch)
            pred_label = logits.argmax(dim=-1).item()

        _sync_if_cuda()
        start = time.perf_counter()
        _ = explain(
            model,
            data.x,
            data.edge_index,
            getattr(data, "edge_attr", None),
            batch,
        )
        _sync_if_cuda()
        end = time.perf_counter()
        times_ms.append((end - start) * 1000.0)

    arr = np.asarray(times_ms, dtype=float)
    mean_ms = float(arr.mean()) if arr.size else float("nan")
    median_ms = float(np.median(arr)) if arr.size else float("nan")
    p90_ms = float(np.percentile(arr, 90)) if arr.size else float("nan")
    total_s = float(arr.sum() / 1000.0) if arr.size else float("nan")
    graphs_per_s = float(1000.0 / mean_ms) if mean_ms > 0 else float("nan")
    return {
        "mean_ms": mean_ms,
        "median_ms": median_ms,
        "p90_ms": p90_ms,
        "total_s": total_s,
        "graphs_per_s": graphs_per_s,
        "n_graphs": int(arr.size),
    }


In [18]:

def run_bcos_experiment(
    seeds=(11, 22, 33),
    batch_size=128,
    hidden_dim=64,
    num_layers=4,
    lr=1e-3,
    max_epochs=100,
    patience=25,
    min_delta=1e-6,
    b=2.0,
    max_out=1,
    dropout=0.3,
):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    criterion = nn.CrossEntropyLoss()
    per_seed = []

    for run_idx, seed in enumerate(seeds, start=1):
        print(f"\n=== Seed {seed} ===")
        set_seed(seed)

        train_loader, val_loader, test_loader, full_dataset, test_dataset = load_and_split_data(
            batch_size=batch_size,
        )

        model = PureBcosGINE(
            node_dim=full_dataset.num_node_features,
            edge_dim=full_dataset.num_edge_features,
            hidden_dim=hidden_dim,
            num_layers=num_layers,
            num_classes=full_dataset.num_classes,
            b=b,
            max_out=max_out,
            dropout=dropout,
        ).to(device)

        # IG uses AdamW with weight decay
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        # IG uses ReduceLROnPlateau
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", factor=0.5, patience=10, min_lr=1e-6
        )

        best_val_loss = float('inf')
        best_state_dict = None
        patience_counter = 0

        _sync_if_cuda()
        train_start = time.perf_counter()

        for epoch in range(1, max_epochs + 1):
            train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
            val_loss, val_acc, val_f1 = evaluate(model, val_loader, criterion, device)

            scheduler.step(val_loss)
            
            if epoch % 5 == 0 or epoch == 1:
                print(f"  Epoch {epoch:03d}/{max_epochs}: train_loss={train_loss:.4f} val_acc={val_acc:.4f} val_loss={val_loss:.4f}")

            if val_loss < best_val_loss - min_delta:
                best_val_loss = val_loss
                best_state_dict = copy.deepcopy(model.state_dict())
                patience_counter = 0
            else:
                patience_counter += 1

            if patience_counter >= patience:
                print(f"  Early stopping at epoch {epoch}")
                break

        _sync_if_cuda()
        train_time_s = time.perf_counter() - train_start

        if best_state_dict is not None:
            model.load_state_dict(best_state_dict)

        test_acc, test_f1, node_auc, node_f1, node_jaccard = evaluate_bcos_dynamic_k(
            model, test_loader, test_dataset, device
        )

        test_process_total_s = benchmark_test_inference_total(model, test_loader, device)
        timing = time_bcos_explanations(model, test_dataset, device, warmup=2)

        row = {
            "seed": seed,
            "test_acc": test_acc,
            "test_f1": test_f1,
            "node_auroc": node_auc,
            "node_f1": node_f1,
            "node_jaccard": node_jaccard,
            "train_time_s": train_time_s,
            "test_process_total_s": test_process_total_s,
            "explain_total_s": timing["total_s"],
            "mean_ms": timing["mean_ms"],
            "median_ms": timing["median_ms"],
            "p90_ms": timing["p90_ms"],
            "graphs_per_s": timing["graphs_per_s"],
            "n_graphs": timing["n_graphs"],
        }
        per_seed.append(row)

        print(
            f"Seed {seed} | Test Acc {test_acc:.4f} | Test F1 {test_f1:.4f} | Node AUROC {node_auc:.4f} | Node F1 {node_f1:.4f} | Node Jaccard {node_jaccard:.4f} | "
            f"Train {train_time_s:.2f}s | TestProc {test_process_total_s:.2f}s | "
            f"B-COS Explain {timing['total_s']:.2f}s ({timing['mean_ms']:.2f} ms/graph)"
        )

    print("\n" + "=" * 84)
    print("MULTI-SEED SUMMARY (mean ± std)")
    print("=" * 84)
    metrics = [
        "test_acc",
        "test_f1",
        "node_auroc",
        "node_f1",
        "node_jaccard",
        "train_time_s",
        "test_process_total_s",
        "explain_total_s",
        "mean_ms",
        "median_ms",
        "p90_ms",
        "graphs_per_s",
    ]
    import numpy as np
    for metric in metrics:
        vals = np.array([row[metric] for row in per_seed], dtype=float)
        print(f"{metric:22s}: {vals.mean():.4f} ± {vals.std():.4f}")

    print("-" * 84)
    print(f"Total explanation time over all seeds : {sum(row['explain_total_s'] for row in per_seed):.4f} s")
    print(f"Total test processing time over all seeds: {sum(row['test_process_total_s'] for row in per_seed):.4f} s")
    print(f"Total training time over all seeds       : {sum(row['train_time_s'] for row in per_seed):.4f} s")

    return per_seed


In [19]:

bcos_results = run_bcos_experiment(
    seeds=(11, 22, 33),
    batch_size=128,
    hidden_dim=64,
    num_layers=4,
    lr=1e-3,
    max_epochs=100,
    patience=25,
    min_delta=1e-6,
    b=2.0,
    max_out=1,
    dropout=0.3,
)

import pandas as pd
bcos_result_df = pd.DataFrame(bcos_results)
bcos_result_df


Using device: cpu

===== B-Cos Run 1/3 | Seed: 42 =====
Loading dataset...
Dataset loaded. Train size: 54000, Val size: 6000, Test size: 10000
Starting training...
Dataset loaded. Train size: 54000, Val size: 6000, Test size: 10000
Starting training...


Epoch 001/100, Train Loss: 2.9460, Val Loss: 2.1290, Val Acc: 0.1945, Val F1: 0.1496, LR: 0.001000


Epoch 002/100, Train Loss: 2.0935, Val Loss: 2.0673, Val Acc: 0.2255, Val F1: 0.1398, LR: 0.000999


Epoch 003/100, Train Loss: 2.0818, Val Loss: 2.0676, Val Acc: 0.2208, Val F1: 0.1463, LR: 0.000998


Epoch 004/100, Train Loss: 2.0529, Val Loss: 1.9528, Val Acc: 0.2637, Val F1: 0.1816, LR: 0.000996


...

Epoch 032/100, Train Loss: 1.0630, Val Loss: 1.0002, Val Acc: 0.6553, Val F1: 0.6539, LR: 0.000768


Epoch 033/100, Train Loss: 1.0517, Val Loss: 0.9937, Val Acc: 0.6697, Val F1: 0.6669, LR: 0.000755


Epoch 034/100, Train Loss: 1.1019, Val Loss: 1.0035, Val Acc: 0.6598, Val F1: 0.6585, LR: 0.000741


Epoch 035/100, Train Loss: 1.0411, Val Loss: 1.0173, Val Acc: 0.6585, Val F1: 0.6521, LR: 0.000727


KeyboardInterrupt: 